In [28]:
import pandas as pd
import numpy as np

In [29]:
df = pd.read_csv("Dataset/matches_clean.csv", parse_dates=["match_date"])
df = df.sort_values(["match_date", "id"]).reset_index(drop=True)

In [30]:
#Reshape the df to have 1 row per team, per match

home = df[["id", "match_date", "season_start_year", "home", "home_score", "away_score"]].copy()
home.columns = ["id", "match_date", "season", "team", "gf", "ga"] 
home["venue"] = "H"

away = df[["id", "match_date", "season_start_year", "away", "away_score", "home_score"]].copy()
away.columns = ["id", "match_date", "season", "team", "gf", "ga"]
away["venue"] = "A"

#Combine the home and away dataframes into a long format dataframe, where both home and away matches are in a single df, sorted by match id and date
long = pd.concat([home, away]).sort_values(["match_date", "id"]).reset_index(drop=True)

long.shape #7967*2 checks out

(15934, 7)

In [31]:
long["pts"]= np.select([long.gf> long.ga, long.gf==long.ga], [3,1], default=0) #points earned by each row
long["win"]= (long.gf>long.ga).astype(int) #win flag

In [32]:
#Recent form (5 matches)
long["recent_5"]= long.groupby("team")["pts"].transform(
    lambda s:s.shift(1).rolling(5, min_periods=5).sum()
)

In [33]:
#average goal scored and conceded
#expanding instead of rolling, as we want to use all previous matches to calculate the average, not just the last 5
long["avg_gf"]= long.groupby("team")["gf"].transform(lambda s:s.shift(1).expanding(min_periods=5).mean())
long["avg_ga"]= long.groupby("team")["ga"].transform(lambda s:s.shift(1).expanding(min_periods=5).mean())

In [34]:
#Home win rate and away win rate separately
long["win_rate"]= long.groupby(["team", "venue"])["win"].transform(
    lambda s:s.shift(1).expanding(min_periods=5).mean()
)

long.tail(25)

,id,match_date,season,team,gf,ga,venue,pts,win,recent_5,avg_gf,avg_ga,win_rate
15909,605717,2022-05-11,2021,Manchester City,5,1,A,3,1,13.0,1.885073,1.025099,0.452381
15910,605744,2022-05-11,2021,Watford,0,0,H,1,0,0.0,1.038023,1.650190,0.305344
15911,605744,2022-05-11,2021,Everton,0,0,A,1,0,10.0,1.323678,1.246851,0.274559
15912,605835,2022-05-11,2021,Leicester City,3,0,H,3,1,2.0,1.412234,1.430851,0.414894
15913,605835,2022-05-11,2021,Norwich City,0,3,A,0,0,3.0,0.950570,1.825095,0.114504
15914,605819,2022-05-12,2021,Tottenham Hotspur,3,0,H,3,1,8.0,1.593711,1.203774,0.584383
15915,605819,2022-05-12,2021,Arsenal,0,3,A,0,0,12.0,1.877987,1.046541,0.463476
15916,605670,2022-05-15,2021,Everton,2,3,H,0,0,8.0,1.322013,1.245283,0.503778
15917,605670,2022-05-15,2021,Brentford,3,2,A,3,1,10.0,1.222222,1.444444,0.277778
15918,605671,2022-05-15,2021,Leeds United,1,1,H,1,0,4.0,1.340426,1.617021,0.351064


check everton. here it had 0.503 for "H" while only 0.27 for "A"

In [35]:
team = "Manchester City"   
a = long[long.team == team].tail(12)[["match_date", "venue", "win", "win_rate"]]
print(a)

      match_date venue  win  win_rate
15680 2022-02-19     H    0  0.651475
15703 2022-02-26     A    1  0.449198
15730 2022-03-06     H    1  0.649733
15761 2022-03-14     A    0  0.450667
15783 2022-04-02     A    1  0.449468
15814 2022-04-10     H    0  0.650667
15832 2022-04-20     H    1  0.648936
15850 2022-04-23     H    1  0.649867
15873 2022-04-30     A    1  0.450928
15900 2022-05-08     H    1  0.650794
15909 2022-05-11     A    1  0.452381
15927 2022-05-15     A    0  0.453826


In [36]:
#For League position vefore the match

pos_rows = []

for season, s_df in long.groupby("season"):
    table = {}   # team -> [points, goal_diff, goals_for, played]

    for date, day in s_df.groupby("match_date"):
        # 1) rank teams using the table BEFORE today's games
        order = sorted(table, key=lambda t: (-table[t][0], -table[t][1], -table[t][2]))
        rank = {team: i + 1 for i, team in enumerate(order)}

        for r in day.itertuples():
            played = table.get(r.team, [0, 0, 0, 0])[3]
            pos_rows.append((r.id, r.venue, rank.get(r.team, np.nan), played))

        # 2) only now add today's results to the table
        for r in day.itertuples():
            t = table.setdefault(r.team, [0, 0, 0, 0])
            t[0] += r.pts
            t[1] += r.gf - r.ga
            t[2] += r.gf
            t[3] += 1

pos = pd.DataFrame(pos_rows, columns=["id", "venue", "position", "played"])
long = long.merge(pos, on=["id", "venue"], how="left")

long.head(5)

,id,match_date,season,team,gf,ga,venue,pts,win,recent_5,avg_gf,avg_ga,win_rate,position,played
0,18123,2001-08-18,2001,Charlton Athletic,1,2,H,0,0,NaN,NaN,NaN,NaN,NaN,0
1,18123,2001-08-18,2001,Everton,2,1,A,3,1,NaN,NaN,NaN,NaN,NaN,0
2,18124,2001-08-18,2001,Derby County,2,1,H,3,1,NaN,NaN,NaN,NaN,NaN,0
3,18124,2001-08-18,2001,Blackburn Rovers,1,2,A,0,0,NaN,NaN,NaN,NaN,NaN,0
4,18125,2001-08-18,2001,Leeds United,2,0,H,3,1,NaN,NaN,NaN,NaN,NaN,0


In [37]:
print(long.columns.tolist())

['id', 'match_date', 'season', 'team', 'gf', 'ga', 'venue', 'pts', 'win', 'recent_5', 'avg_gf', 'avg_ga', 'win_rate', 'position', 'played']


In [38]:
cols = ["recent_5", "avg_gf", "avg_ga", "win_rate", "position", "played"]

home_feats = long[long.venue == "H"][["id"] + cols].rename(columns={c: "home_" + c for c in cols})
away_feats = long[long.venue == "A"][["id"] + cols].rename(columns={c: "away_" + c for c in cols})

data = df[["id", "match_date", "season_start_year", "home", "away", "target"]]
data = data.merge(home_feats, on="id").merge(away_feats, on="id")

In [41]:
print(data.columns.tolist())

['id', 'match_date', 'season_start_year', 'home', 'away', 'target', 'home_recent_5', 'home_avg_gf', 'home_avg_ga', 'home_win_rate', 'home_position', 'home_played', 'away_recent_5', 'away_avg_gf', 'away_avg_ga', 'away_win_rate', 'away_position', 'away_played', 'position_diff']


In [42]:
data["position_diff"] = data["away_position"] - data["home_position"]   # positive = home team is higher
data["form_diff"] = data["home_recent_5"] - data["away_recent_5"]
data["target_encoded"] = data["target"].map({"A": 0, "D": 1, "H": 2})

In [44]:
#Drop rows with missing values in any of the feature columns, as we cannot use them for training
feature_cols = [
    "home_recent_5", "away_recent_5", "position_diff",
    "home_win_rate", "away_win_rate",
    "home_avg_gf", "home_avg_ga", "away_avg_gf", "away_avg_ga",
    "home_position", "away_position", "position_diff",
]

print("Before:", len(data))
ml_data = data.dropna(subset=feature_cols).reset_index(drop=True)
print("After :", len(ml_data))

ml_data.to_csv("Dataset/features.csv", index=False)

Before: 7967
After : 7464


In [72]:
row = ml_data.iloc[19]
print(row[["match_date", "home", "home_recent_5"]])

team = row["home"]
past = long[(long.team == team) & (long.match_date < row["match_date"])].tail(5)
print(past[["match_date", "gf", "ga", "pts"]])
print("Manual form:", past["pts"].sum())

match_date       2001-11-24 00:00:00
home                         Chelsea
home_recent_5                    6.0
Name: 19, dtype: object
    match_date  gf  ga  pts
177 2001-10-21   0   0    1
181 2001-10-24   1   2    0
203 2001-10-28   1   1    1
224 2001-11-04   2   1    3
241 2001-11-18   0   0    1
Manual form: 6


In [48]:
#checking if all feature columns are present in the ml_data . output [] if present
print([c for c in feature_cols if c not in ml_data.columns])

[]


In [50]:
#Final sanity checks. since we dropped rows with missing values, all feature columns should have 0 missing values, and the target_encoded column should have a reasonable distribution of classes
print(ml_data[feature_cols].isna().sum())   # should all be 0
print(ml_data["target_encoded"].value_counts(normalize=True))

home_recent_5    0
away_recent_5    0
position_diff    0
home_win_rate    0
away_win_rate    0
home_avg_gf      0
home_avg_ga      0
away_avg_gf      0
away_avg_ga      0
home_position    0
away_position    0
position_diff    0
dtype: int64
target_encoded
2    0.459137
0    0.292471
1    0.248392
Name: proportion, dtype: float64
